In [1]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB  
from haversine import haversine

In [2]:
import kagglehub

# Download the latest version
path = kagglehub.dataset_download("fatihb/citibike-sampled-data-2013-2017")

print("Path to dataset files:", path)

Path to dataset files: /Users/michaellukas/.cache/kagglehub/datasets/fatihb/citibike-sampled-data-2013-2017/versions/1


In [5]:
#Reading values from csv files
trips = pd.read_csv(f"{path}/citibike-trips.csv")
stations = pd.read_csv(f"{path}/citibike-stations.csv")
print(trips)
print(stations)

        tripduration            starttime             stoptime  \
0               2319  2016-03-09 13:08:21  2016-03-09 13:47:01   
1                313  2015-07-09 15:42:30  2015-07-09 15:47:44   
2                798  2017-04-20 18:43:59  2017-04-20 18:57:17   
3               3100  2017-04-23 15:23:46  2017-04-23 16:15:26   
4                906  2016-01-11 18:32:33  2016-01-11 18:47:39   
...              ...                  ...                  ...   
473551          1368  2014-03-31 16:08:18  2014-03-31 16:31:06   
473552          1283  2015-07-26 12:47:07  2015-07-26 13:08:31   
473553           620  2016-06-18 15:18:37  2016-06-18 15:28:58   
473554          1510  2017-05-20 15:51:49  2017-05-20 16:16:59   
473555          1358  2015-09-05 14:20:13  2015-09-05 14:42:51   

        start_station_id        start_station_name  start_station_latitude  \
0                    520           W 52 St & 5 Ave               40.759923   
1                    520           W 52 St & 5 Ave 

In [7]:
# Convert start time and stop time to datetime objects 
trips["starttime"] = pd.to_datetime(trips["starttime"])
trips["stoptime"]  = pd.to_datetime(trips["stoptime"])

# Set up evaluation windows
windows = {
    "AM": {"start_hour": 8, "end_hour": 9},    # 8–9 AM
    "PM": {"start_hour": 16, "end_hour": 18}   # 4–6 PM
}

# List of station IDs and capcities
stations_list = stations['station_id_int'].dropna().astype(int).tolist()
C = dict(zip(stations['station_id_int'], stations['capacity']))


# Build per-station, per-window demand data 
window_data = {
    "AM": {"D": {}, "R": {}, "N": {}},
    "PM": {"D": {}, "R": {}, "N": {}}
}

# initalizing to zero for every station
for w in window_data.keys():
    for sid in stations_list:
        window_data[w]["D"][sid] = 0
        window_data[w]["R"][sid] = 0
        window_data[w]["N"][sid] = 0

# Add hours columns 
trips['start_hour'] = trips['starttime'].dt.hour
trips['stop_hour'] = trips['stoptime'].dt.hour

# Aggregate deamnd for each window 
for w, config in windows.items():
    sh = config["start_hour"] # Start hour
    eh = config["end_hour"] # End hour

    dep_mask = (trips['start_hour'] >= sh) & (trips['start_hour'] < eh) # Picking trips that start in that hour window
    ret_mask = (trips['stop_hour'] >= sh) & (trips['stop_hour'] < eh) # Picking trips that end in that hour window

    dep = trips.loc[dep_mask].groupby('start_station_id').size()
    ret = trips.loc[ret_mask].groupby('end_station_id').size()

# Initializing stations that dont have trips in that window 
    for sid in stations_list:
        D_i = int(dep.get(sid, 0))
        R_i = int(ret.get(sid, 0))
        N_i = D_i - R_i
        window_data[w]["D"][sid] = D_i
        window_data[w]["R"][sid] = R_i
        window_data[w]["N"][sid] = N_i

# Starting bikes before rebalancing (Bi from the doc)
# Since data set doesn't tell us how many bikes were at a station at exactly 8:00 AM on a typical day
# We will assume the stations start half full - 50% of capacity
B_AM = {sid: min(C[sid], int(0.5 * C[sid])) for sid in stations_list}
B_PM = {sid: min(C[sid], int(0.5 * C[sid])) for sid in stations_list}


In [9]:
def run_rebalancing_model(MOVE_BUDGET, fill_level):
    import gurobipy as gp
    from gurobipy import GRB

    # ---- Build starting bikes using fill_level ----
    B_AM = {sid: min(C[sid], int(fill_level * C[sid])) for sid in stations_list}
    B_PM = {sid: min(C[sid], int(fill_level * C[sid])) for sid in stations_list}

    # ---- Create model ----
    m = gp.Model("Citibike_Rebalance_Peak")

    windows = ["AM", "PM"]

    # Decision vars
    num_bikes = {
        w: m.addVars(stations_list, vtype=GRB.INTEGER, lb=0, name=f"b_{w}")
        for w in windows
    }

    short_slack = {
        w: m.addVars(stations_list, vtype=GRB.INTEGER, lb=0, name=f"s_short_{w}")
        for w in windows
    }
    over_slack = {
        w: m.addVars(stations_list, vtype=GRB.INTEGER, lb=0, name=f"s_over_{w}")
        for w in windows
    }

    move_plus = {
        w: m.addVars(stations_list, lb=0, name=f"move_in_{w}")
        for w in windows
    }
    move_minus = {
        w: m.addVars(stations_list, lb=0, name=f"move_out_{w}")
        for w in windows
    }

    # Objective
    m.setObjective(
        gp.quicksum(short_slack[w][i] + over_slack[w][i]
                    for w in windows for i in stations_list),
        GRB.MINIMIZE
    )

    # Constraints
    for w in windows:
        for i in stations_list:
            N_i = window_data[w]["N"][i]

            start_B = B_AM[i] if w == "AM" else B_PM[i]

            # linking constraint
            m.addConstr(
                num_bikes[w][i] == start_B + move_plus[w][i] - move_minus[w][i],
                name=f"link_{w}_{i}"
            )

            m.addConstr(short_slack[w][i] >= N_i - num_bikes[w][i])
            m.addConstr(over_slack[w][i] >= num_bikes[w][i] - N_i - C[i])

            m.addConstr(num_bikes[w][i] <= C[i])
            m.addConstr(num_bikes[w][i] >= 0)

        # total move budget constraint
        m.addConstr(
            gp.quicksum(move_plus[w][i] + move_minus[w][i] for i in stations_list)
            <= MOVE_BUDGET,
            name=f"move_budget_{w}"
        )

    # Optimize
    m.optimize()

    # Return all useful objects
    return (
        m.objVal,
        num_bikes,
        short_slack,
        over_slack,
        move_plus,
        move_minus,
    )


In [15]:
# Sensitivity Analysis 
# 1: Evaluate effect of MOVE_BUDGET

budgets = [200, 500, 1000, 2000, 4000, 8000]
results_budget = []

for M in budgets:
    obj, _, _, _, _, _ = run_rebalancing_model(M, fill_level=0.50)
    results_budget.append((M, obj))
    print(f"MOVE_BUDGET = {M:4d} → objective = {obj:,.0f}")


Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 23.5.0 23F79)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 18022 rows, 18020 columns and 39644 nonzeros
Model fingerprint: 0xaf91655c
Variable types: 7208 continuous, 10812 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 5e+02]
Found heuristic solution: objective 21876.000000
Presolve removed 15825 rows and 12356 columns
Presolve time: 0.09s
Presolved: 2197 rows, 5664 columns, 9594 nonzeros
Found heuristic solution: objective 21588.000000
Variable types: 3468 continuous, 2196 integer (69 binary)

Root relaxation: objective 2.138200e+04, 2641 iterations, 0.02 seconds (0.02 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It

In [17]:
def solve_model_with_fill(alpha, move_budget=500):
    """
    alpha in [0,1]: fraction of capacity each station starts with.
    move_budget: truck move budget M (same for AM & PM).
    Returns objective value (total service failures).
    """
    # Build starting-bike dictionaries using alpha
    B_AM = {sid: min(C[sid], int(alpha * C[sid])) for sid in stations_list}
    B_PM = {sid: min(C[sid], int(alpha * C[sid])) for sid in stations_list}

    # New model
    m = gp.Model("Citibike_fill_sensitivity")
    windows = ["AM", "PM"]

    # Decision vars
    num_bikes = {
        "AM": m.addVars(stations_list, vtype=GRB.INTEGER, lb=0.0, name="b_AM"),
        "PM": m.addVars(stations_list, vtype=GRB.INTEGER, lb=0.0, name="b_PM"),
    }
    short_slack = {
        "AM": m.addVars(stations_list, vtype=GRB.INTEGER, lb=0.0, name="s_short_AM"),
        "PM": m.addVars(stations_list, vtype=GRB.INTEGER, lb=0.0, name="s_short_PM"),
    }
    over_slack = {
        "AM": m.addVars(stations_list, vtype=GRB.INTEGER, lb=0.0, name="s_over_AM"),
        "PM": m.addVars(stations_list, vtype=GRB.INTEGER, lb=0.0, name="s_over_PM"),
    }
    move_plus = {
        "AM": m.addVars(stations_list, lb=0, name="move_in_AM"),
        "PM": m.addVars(stations_list, lb=0, name="move_in_PM"),
    }
    move_minus = {
        "AM": m.addVars(stations_list, lb=0, name="move_out_AM"),
        "PM": m.addVars(stations_list, lb=0, name="move_out_PM"),
    }

    # Objective
    m.setObjective(
        gp.quicksum(short_slack[w][i] + over_slack[w][i]
                    for w in windows for i in stations_list),
        GRB.MINIMIZE
    )

    # Constraints
    for w in windows:
        for i in stations_list:
            N_i = window_data[w]["N"][i]
            start_B = B_AM[i] if w == "AM" else B_PM[i]

            # linking
            m.addConstr(num_bikes[w][i] == start_B + move_plus[w][i] - move_minus[w][i])

            # shortage / overflow
            m.addConstr(short_slack[w][i] >= N_i - num_bikes[w][i])
            m.addConstr(over_slack[w][i]  >= num_bikes[w][i] - N_i - C[i])

            # capacity
            m.addConstr(num_bikes[w][i] <= C[i])
            m.addConstr(num_bikes[w][i] >= 0)

        # move budget for that window
        m.addConstr(
            gp.quicksum(move_plus[w][i] + move_minus[w][i] for i in stations_list) <= move_budget,
            name=f"move_budget_{w}"
        )

    m.setParam("OutputFlag", 0)   # silence solver
    m.optimize()
    return m.objVal


In [19]:
fill_levels = [0.3, 0.5, 0.7, 0.9]   # 30%, 50%, 70%, 90%
move_budget_fixed = 500

for alpha in fill_levels:
    obj = solve_model_with_fill(alpha, move_budget=move_budget_fixed)
    print(f"alpha = {alpha:.1f} → failures = {obj:,.0f}")


alpha = 0.3 → failures = 21,924
alpha = 0.5 → failures = 20,782
alpha = 0.7 → failures = 20,596
alpha = 0.9 → failures = 21,551


In [23]:
# === Baseline model for heuristic sensitivity ===

BASE_ALPHA = 0.5       # 50% starting fill
MOVE_BUDGET = 500      # same budget we used as baseline
windows = ["AM", "PM"]

# Starting bikes B_AM, B_PM
B_AM = {sid: min(C[sid], int(BASE_ALPHA * C[sid])) for sid in stations_list}
B_PM = {sid: min(C[sid], int(BASE_ALPHA * C[sid])) for sid in stations_list}

m = gp.Model("Citibike_baseline_for_heuristic")

# Decision variables
num_bikes = {
    "AM": m.addVars(stations_list, vtype=GRB.INTEGER, lb=0.0, name="b_AM"),
    "PM": m.addVars(stations_list, vtype=GRB.INTEGER, lb=0.0, name="b_PM"),
}
short_slack = {
    "AM": m.addVars(stations_list, vtype=GRB.INTEGER, lb=0.0, name="s_short_AM"),
    "PM": m.addVars(stations_list, vtype=GRB.INTEGER, lb=0.0, name="s_short_PM"),
}
over_slack = {
    "AM": m.addVars(stations_list, vtype=GRB.INTEGER, lb=0.0, name="s_over_AM"),
    "PM": m.addVars(stations_list, vtype=GRB.INTEGER, lb=0.0, name="s_over_PM"),
}
move_plus = {
    "AM": m.addVars(stations_list, lb=0, name="move_in_AM"),
    "PM": m.addVars(stations_list, lb=0, name="move_in_PM"),
}
move_minus = {
    "AM": m.addVars(stations_list, lb=0, name="move_out_AM"),
    "PM": m.addVars(stations_list, lb=0, name="move_out_PM"),
}

# Objective: total service failures
m.setObjective(
    gp.quicksum(short_slack[w][i] + over_slack[w][i]
                for w in windows for i in stations_list),
    GRB.MINIMIZE
)

# Constraints
for w in windows:
    for i in stations_list:
        N_i = window_data[w]["N"][i]
        start_B = B_AM[i] if w == "AM" else B_PM[i]

        m.addConstr(num_bikes[w][i] == start_B + move_plus[w][i] - move_minus[w][i])
        m.addConstr(short_slack[w][i] >= N_i - num_bikes[w][i])
        m.addConstr(over_slack[w][i]  >= num_bikes[w][i] - N_i - C[i])
        m.addConstr(num_bikes[w][i] <= C[i])
        m.addConstr(num_bikes[w][i] >= 0)

    m.addConstr(
        gp.quicksum(move_plus[w][i] + move_minus[w][i] for i in stations_list) <= MOVE_BUDGET,
        name=f"move_budget_{w}"
    )

m.optimize()
print("Baseline objective =", m.objVal)


Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 23.5.0 23F79)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 18022 rows, 18020 columns and 39644 nonzeros
Model fingerprint: 0xb871c276
Variable types: 7208 continuous, 10812 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 5e+02]
Found heuristic solution: objective 21780.000000
Presolve removed 15825 rows and 12356 columns
Presolve time: 0.11s
Presolved: 2197 rows, 5664 columns, 9594 nonzeros
Found heuristic solution: objective 21270.000000
Variable types: 3468 continuous, 2196 integer (69 binary)

Root relaxation: objective 2.078200e+04, 2619 iterations, 0.01 seconds (0.02 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It

In [25]:
# Building an algorithm for our "greedy" van
import math

# Build task list for AM window based on our main model
pickups_AM = []
drops_AM = []

# Finding q_out (bikes to remove) and q_in (bikes to deliver)
for sid in stations_list:
    q_out = int(round(move_minus["AM"][sid].X)) # bikes trucked into station sid in window w
    q_in = int(round(move_plus["AM"][sid].X)) # bikes trucked out of station sid in window w

    if q_out == 0 and q_in == 0:
        continue # skipping stations that have no moves

    # Get coordinates from stations dataFrame
    row = stations.loc[stations['station_id_int'] == sid].iloc[0] # Get the first row of the DF
    lat = float(row['latitude'])
    lon = float(row['longitude'])

    if q_out > 0:
        pickups_AM.append((sid, q_out, lat, lon))
    if q_in > 0:
       drops_AM.append((sid, q_in, lat, lon))

print(f"AM Pickups: {len(pickups_AM)} stations")
print(f"AM Drops: {len(drops_AM)} stations")
pickups_AM[:5], drops_AM[:5]

# Interpreting output, ex. at station 3233 the model wants to us pick up 17 bikes
# At station 399 the model wants us to drop 11 bikes (So it's short 11 bikes)
# At this point we have out list for the van and now need to figure out in what order

AM Pickups: 17 stations
AM Drops: 13 stations


([(3233, 17, 40.75724567911726, -73.97805914282799),
  (427, 34, 40.701907, -74.013942),
  (362, 3, 40.75172632, -73.98753523),
  (3086, 5, 40.715143, -73.944507),
  (323, 5, 40.69236178, -73.98631746)],
 [(399, 11, 40.68851534, -73.9647628),
  (3077, 10, 40.70877084, -73.95095259),
  (482, 16, 40.73935542, -73.99931783),
  (349, 31, 40.71850211, -73.98329859),
  (339, 21, 40.72580614, -73.97422494)])

In [27]:
# Function to calculate travel time
def travel_minutes(lat1, lon1, lat2, lon2, speed_kmh=20):
    dist_km = haversine((lat1, lon1), (lat2, lon2))
    return dist_km/ (speed_kmh/60) # Convert the speed to km/min 

In [29]:
# Now we can creatre a function for the van route 
def greedy_van_route(start_lat, start_lon, 
                     pickups, drops, capacity = 16, 
                     time_budget_min = 120, speed_kmh=20):

    # Starting by building remaining demand dictionaries and location lookups for pickups/drops
    remaining_pickups = {sid: qty for sid, qty, lat, lon in pickups} # Dict of how many bikes need to be removed
    pickup_locs = {sid: (lat,lon) for sid, qty, lat, lon in pickups} # Dict of the station locations for each sid

    remaining_drops = {sid: qty for sid, qty, lat, lon in drops}
    drop_locs = {sid: (lat,lon) for sid, qty, lat, lon in drops}
    
    # Define what we must track as the van moves - i.e. the van sate
    cur_lat, cur_lon = start_lat, start_lon # Where is the van currently
    load = 0 # How many bikes are on the van (none to start)
    time_used = 0.0 # How much time has passed (minutes)
    route = [] # list of stops

    # Main loop which runs while we have demand and/or time remaining
    while time_used < time_budget_min:
        
        # Decide to go to a pickup or a drop off next
        total_pickups_needed = sum(remaining_pickups.values())
        total_drops_needed = sum(remaining_drops.values())

        if load == 0 and total_pickups_needed > 0:
            next_type = 'pickup' # Van is empty and pickups are needed, go pick up
        elif load == 0 and total_pickups_needed == 0:
            break # Nothing to pickup and van is empty means we're done
        elif load == capacity and total_drops_needed > 0:
            next_type = 'drop' # Van full and there are drops needed, go drop bikes
        elif total_drops_needed == 0 and total_pickups_needed > 0:
            next_type = 'pickup' # Only pickups left, go pick up
        elif total_pickups_needed == 0 and total_drops_needed > 0:
            next_type = 'drop' # Only drops left, go drop
        else:
            # tie breaker: we have pickups and drops remaining and 0<load<capacity, prefer dropping
            next_type = 'drop' if load > 0 else 'pickup'

        # Build a list of candidate stations and choose the nearest one
        if next_type == 'pickup':
            candidate_ids = [sid for sid, qty in remaining_pickups.items() if qty > 0]
            locs = pickup_locs
        else:
            candidate_ids = [sid for sid, qty in remaining_drops.items() if qty > 0]
            locs = drop_locs

        best_sid = None # Initialize best station ID
        best_travel = None # Initialize shortest travel time (in mins)

        for sid in candidate_ids:
            lat2, lon2 = locs[sid] # Look up the station coordinates
            # calculate time it takes to drive from the current van position to this station
            travel = travel_minutes(cur_lat, cur_lon, lat2, lon2, speed_kmh) 

            # keep the station with the minimal travel time
            if time_used + travel <= time_budget_min:
                if (best_travel is None) or (travel<best_travel):
                    best_travel = travel
                    best_sid = sid

        if best_sid is None:
            break # No feasible stop within remaining time, break

        # Moving to the selected station and updating time and location
        time_used += best_travel
        cur_lat, cur_lon = locs[best_sid]

        # Decide how many bikes to move at this station
        if next_type == 'pickup':
            bikes_need = remaining_pickups[best_sid] # How many bikes needed at the station
            can_take = min(bikes_need, capacity-load) # Can only take at most whats needed and what fits in van
            moved = can_take
            load += moved # Update current van load
            remaining_pickups[best_sid] -= moved # Subtract how many we moved from rem_pickups
            if remaining_pickups[best_sid] <= 0:
                del remaining_pickups[best_sid]
        else: # Drop case
            bikes_need = remaining_drops[best_sid]
            can_drop = min(bikes_need, load)
            moved = can_drop
            load -= moved # update current van load
            remaining_drops[best_sid] -= moved
            if remaining_drops[best_sid] <= 0:
                del remaining_drops[best_sid]

        if moved == 0:
            break # If we cant move bikes (load = 0 at drop off), we break to avoid an infinite loop

        # Now we can update Route to log our movements
        route.append({
            'station_id': best_sid,
            'action': next_type,
            'bikes_moved': moved,
            'arrival_min': round(time_used, 1), # Gives us the approx time of arrival
            'time_remaining': round(time_budget_min - time_used, 1),
            'van_load_after': load # Van load after the move 
        })

    return route, time_used, load

In [37]:
# Sensitivity 3: effect of different time windows for the greedy van

time_windows = [60, 90, 120]

for T in time_windows:
    route_T, time_used_T, final_load_T = greedy_van_route(
        start_lat, start_lon,
        pickups_AM, drops_AM,
        capacity=16,
        time_budget_min=T,
        speed_kmh=20
    )
    
    bikes_moved_T = sum(step["bikes_moved"] for step in route_T)
    time_remaining_T = T - time_used_T

    print(
        f"Time window = {T} min → "
        f"stops = {len(route_T)}, "
        f"bikes moved = {bikes_moved_T}, "
        f"time used = {time_used_T:.1f} min, "
        f"time remaining = {time_remaining_T:.1f} min"
    )


Time window = 60 min → stops = 18, bikes moved = 178, time used = 56.0 min, time remaining = 4.0 min
Time window = 90 min → stops = 25, bikes moved = 255, time used = 86.6 min, time remaining = 3.4 min
Time window = 120 min → stops = 30, bikes moved = 301, time used = 118.4 min, time remaining = 1.6 min
